# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`/work/data`)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Les services

| Adresse | Service |
|---|---|
| http://localhost:8888 | JupyterLab |
| http://localhost:8080 | Master Spark |
| http://localhost:4040 | Spark UI de la session ouverte |
| http://localhost:18080 | Spark History |
| `mongodb://mongo:27017` | MongoDB |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [2]:
schema_produits = T.StructType([
    T.StructField("produit_id", T.StringType()),
    T.StructField("nom", T.StringType()),
    T.StructField("categorie", T.StringType()),
    T.StructField("sous_categorie", T.StringType()),
    T.StructField("prix", T.DoubleType()),
    T.StructField("poids_kg", T.DoubleType())
])

schema_commandes = T.StructType([
    T.StructField("commande_id", T.StringType()),
    T.StructField("client_id", T.StringType()),
    T.StructField("horodatage", T.TimestampType()),
    T.StructField("evenement", T.StringType()),
    T.StructField("livreur_id", T.StringType()),
    T.StructField("detail", T.StringType())
])

schema_communes = T.StructType([
    T.StructField("commune", T.StringType()),
    T.StructField("code_postal", T.StringType()),
    T.StructField("lon", T.DoubleType()),
    T.StructField("lat", T.DoubleType())
])

schema_gps = T.StructType([
    T.StructField("timestamp", T.TimestampType()),
    T.StructField("livreur_id", T.StringType()),
    T.StructField("lon", T.DoubleType()),
    T.StructField("lat", T.DoubleType()),
    T.StructField("vitesse_kmh", T.DoubleType()),
    T.StructField("dans_vehicule", T.BooleanType()),
    T.StructField("charge_kg", T.DoubleType())
])

fmt = "yyyy-MM-dd HH:mm:ss"
produits = spark.read.csv(DATA + "/produits.csv", header=True, schema=schema_produits)
journal = spark.read.csv(DATA + "/commandes.csv", header=True, schema=schema_commandes, timestampFormat=fmt)
communes = spark.read.csv(DATA + "/communes.csv", header=True, schema=schema_communes)
gps = spark.read.csv(DATA + "/vehicules/*/*.csv", header=True, schema=schema_gps, timestampFormat=fmt)

journal.cache()

print("produits :", produits.count())
print("journal :", journal.count())
print("communes :", communes.count())
print("gps :", gps.count())

produits : 233


journal : 846535
communes : 142


gps : 5174905


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [3]:
# le chemin est du type .../vehicules/V01/2025-01.csv
gps = gps.withColumn("vehicule_id", F.regexp_extract(F.input_file_name(), "vehicules/([^/]+)/", 1))
gps = gps.withColumn("jour", F.to_date("timestamp"))
gps.show(5)

+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|          timestamp|livreur_id|     lon|      lat|vitesse_kmh|dans_vehicule|charge_kg|vehicule_id|      jour|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|2025-10-01 07:40:00|      L001|2.277083|48.936735|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:41:00|      L001|2.277083|48.936735|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:42:00|      L001|2.277084|48.936734|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:43:00|      L001|2.277083|48.936734|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:44:00|      L001|2.277084|48.936733|        0.0|        false|   2581.6|        V18|2025-10-01|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
o

## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [4]:
print("commandes :", journal.select("commande_id").distinct().count())
print("clients :", journal.select("client_id").distinct().count())
print("livreurs :", journal.filter(F.col("livreur_id").isNotNull()).select("livreur_id").distinct().count())
print("vehicules :", gps.select("vehicule_id").distinct().count())

commandes : 271617


clients : 8000


livreurs : 90


vehicules : 72


2.2) Comptez les événements du journal par type.

In [5]:
journal.groupBy("evenement").count().orderBy("count", ascending=False).show()

+---------+------+
|evenement| count|
+---------+------+
|  chargee|286862|
| commande|271617|
|   livree|268682|
|    echec| 11220|
| reportee|  6960|
|   retour|  1194|
+---------+------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [6]:
chargees = journal.withColumn("est_chargee", F.when(F.col("evenement") == "chargee", 1).otherwise(0))
nb_chargee = chargees.groupBy("commande_id").agg(F.sum("est_chargee").alias("nb_chargee"))
nb_chargee.groupBy("nb_chargee").count().orderBy("nb_chargee").show()

+----------+------+
|nb_chargee| count|
+----------+------+
|         0|  1637|
|         1|254229|
|         2| 14673|
|         3|  1026|
|         4|    51|
|         5|     1|
+----------+------+



2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [7]:
w_cmd = Window.partitionBy("commande_id").orderBy("horodatage")
avec_prec = journal.withColumn("precedent", F.lag("evenement").over(w_cmd))
avec_prec.filter(F.col("evenement") == "chargee").groupBy("precedent").count().orderBy("count", ascending=False).show()

+---------+------+
|precedent| count|
+---------+------+
| commande|269980|
|    echec|  9950|
| reportee|  6932|
+---------+------+



*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [8]:
livrees = journal.filter(F.col("evenement") == "livree").select("commande_id").distinct()
retours = journal.filter(F.col("evenement") == "retour").select("commande_id").distinct()

toutes = journal.select("commande_id").distinct()
non_livrees = toutes.join(livrees, "commande_id", "left_anti")
abandonnees = non_livrees.join(retours, "commande_id")
en_cours = non_livrees.join(retours, "commande_id", "left_anti")

print("non livrees :", non_livrees.count())
print("abandonnees :", abandonnees.count())
print("en cours :", en_cours.count())

non livrees : 2935


abandonnees : 1194


en cours : 1741


## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [9]:
lignes = journal.filter(F.col("evenement") == "commande")
lignes = lignes.withColumn("produit", F.explode(F.split("detail", "[|]")))
lignes = lignes.withColumn("produit_id", F.split("produit", "x").getItem(0))
lignes = lignes.withColumn("quantite", F.split("produit", "x").getItem(1).cast("int"))
lignes = lignes.select("commande_id", "client_id", "horodatage", "produit_id", "quantite")

lignes.show(5)
print(lignes.count())

+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|         horodatage|produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0037|       5|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0063|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0019|       2|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0067|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0069|       2|
+-----------+---------+-------------------+----------+--------+
only showing top 5 rows



4779049


3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [10]:
lignes = lignes.join(produits, "produit_id")
lignes = lignes.withColumn("montant", F.col("prix") * F.col("quantite"))
lignes.cache()

commandes = lignes.groupBy("commande_id", "client_id", "horodatage").agg(
    F.round(F.sum("montant"), 2).alias("montant"),
    F.round(F.sum(F.col("poids_kg") * F.col("quantite")), 2).alias("poids_kg"),
    F.count("produit_id").alias("nb_lignes")
)
commandes.cache()

commandes.show(5)
print(commandes.count())

+-----------+---------+-------------------+-------+--------+---------+
|commande_id|client_id|         horodatage|montant|poids_kg|nb_lignes|
+-----------+---------+-------------------+-------+--------+---------+
|   C0000168|  CL06333|2025-01-01 13:16:07| 209.35|   60.98|       32|
|   C0000469|  CL03079|2025-01-01 22:39:48|   80.9|   14.03|       13|
|   C0000752|  CL00230|2025-01-02 16:34:53| 218.05|   46.52|       31|
|   C0000810|  CL07244|2025-01-02 18:29:22| 239.45|    78.5|       27|
|   C0000855|  CL04907|2025-01-02 19:48:36|   70.9|   15.87|       15|
+-----------+---------+-------------------+-------+--------+---------+
only showing top 5 rows



271617


## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [11]:
w1 = Window.partitionBy("vehicule_id", "jour").orderBy("timestamp")
w2 = Window.partitionBy("vehicule_id", "jour").orderBy(F.col("timestamp").desc())

tmp = gps.withColumn("n1", F.row_number().over(w1)).withColumn("n2", F.row_number().over(w2))
premiers = tmp.filter("n1 = 1").withColumn("lon_r", F.round("lon", 2)).withColumn("lat_r", F.round("lat", 2))
derniers = tmp.filter("n2 = 1").withColumn("lon_r", F.round("lon", 2)).withColumn("lat_r", F.round("lat", 2))
premiers.cache()

p = premiers.groupBy("lon_r", "lat_r").agg(F.count("*").alias("nb_premiers"))
d = derniers.groupBy("lon_r", "lat_r").agg(F.count("*").alias("nb_derniers"))
positions = p.join(d, ["lon_r", "lat_r"], "outer")
positions.show()
print("nb entrepots :", positions.count())

+-----+-----+-----------+-----------+
|lon_r|lat_r|nb_premiers|nb_derniers|
+-----+-----+-----------+-----------+
| 2.28|48.94|       3208|       3208|
| 2.35|48.75|       2967|       2967|
| 2.45|48.91|       4217|       4217|
+-----+-----+-----------+-----------+



nb entrepots : 3


4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [12]:
ent = premiers.groupBy("lon_r", "lat_r").agg(F.avg("lon").alias("lon"), F.avg("lat").alias("lat"))
ent = ent.withColumn("entrepot", F.concat(F.lit("E"), F.row_number().over(Window.orderBy("lon"))))

entrepots = ent.select("entrepot", "lon", "lat")
entrepots.cache()
entrepots.show()

vehicule_entrepot = premiers.join(ent, ["lon_r", "lat_r"]).select("vehicule_id", "entrepot").distinct()
vehicule_entrepot.groupBy("entrepot").count().orderBy("entrepot").show()

+--------+------------------+------------------+
|entrepot|               lon|               lat|
+--------+------------------+------------------+
|      E1|2.2770689133416475| 48.93671290461345|
|      E2|2.3515135942028986| 48.74704698584426|
|      E3|2.4489591059995255|48.906701645482585|
+--------+------------------+------------------+



+--------+-----+
|entrepot|count|
+--------+-----+
|      E1|   24|
|      E2|   24|
|      E3|   24|
+--------+-----+



## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

In [13]:
w = Window.partitionBy("livreur_id", "jour").orderBy("timestamp")

gps = gps.withColumn("avant", F.lag("dans_vehicule").over(w))
gps = gps.withColumn("changement", F.when((F.col("avant") == True) & (F.col("dans_vehicule") == False), 1).otherwise(0))
gps = gps.drop("avant")

gps.filter((F.col("livreur_id") == "L001") & (F.col("jour") == "2025-01-06") & (F.col("changement") == 1)).show(50)

+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+----------+
|          timestamp|livreur_id|     lon|      lat|vitesse_kmh|dans_vehicule|charge_kg|vehicule_id|      jour|changement|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+----------+
|2025-01-06 09:54:00|      L001|2.311953|48.871389|        0.0|        false|   1143.9|        V10|2025-01-06|         1|
|2025-01-06 10:00:00|      L001|2.313447|48.870848|        0.0|        false|   1089.1|        V10|2025-01-06|         1|
|2025-01-06 10:09:00|      L001|2.316012|48.872919|        0.0|        false|   1085.5|        V10|2025-01-06|         1|
|2025-01-06 10:23:00|      L001|2.325412|48.874465|        0.0|        false|    959.0|        V10|2025-01-06|         1|
|2025-01-06 10:36:00|      L001| 2.32844|48.866695|        0.0|        false|    916.7|        V10|2025-01-06|         1|
|2025-01-06 10:54:00|   

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

In [14]:
gps = gps.withColumn("numero_livraison", F.sum("changement").over(w))
gps.cache()

l001 = gps.filter((F.col("livreur_id") == "L001") & (F.col("jour") == "2025-01-06"))
l001.groupBy("numero_livraison").agg(
    F.min("timestamp").alias("premier"),
    F.max("timestamp").alias("dernier"),
    F.sum(F.when(F.col("dans_vehicule") == False, 1).otherwise(0)).alias("minutes_a_pied")
).orderBy("numero_livraison").show(50)

+----------------+-------------------+-------------------+--------------+
|numero_livraison|            premier|            dernier|minutes_a_pied|
+----------------+-------------------+-------------------+--------------+
|               0|2025-01-06 07:43:00|2025-01-06 09:53:00|            44|
|               1|2025-01-06 09:54:00|2025-01-06 09:59:00|             4|
|               2|2025-01-06 10:00:00|2025-01-06 10:08:00|             7|
|               3|2025-01-06 10:09:00|2025-01-06 10:22:00|            10|
|               4|2025-01-06 10:23:00|2025-01-06 10:35:00|             4|
|               5|2025-01-06 10:36:00|2025-01-06 10:53:00|             8|
|               6|2025-01-06 10:54:00|2025-01-06 11:11:00|             4|
|               7|2025-01-06 11:12:00|2025-01-06 11:37:00|            18|
|               8|2025-01-06 11:38:00|2025-01-06 12:01:00|            17|
|               9|2025-01-06 12:02:00|2025-01-06 12:14:00|             4|
|              10|2025-01-06 12:15:00|

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

In [15]:
a_pied = gps.filter(F.col("dans_vehicule") == False)

livraisons = a_pied.groupBy("livreur_id", "jour", "numero_livraison").agg(
    F.first("vehicule_id").alias("vehicule_id"),
    F.min("timestamp").alias("debut"),
    F.max("timestamp").alias("fin"),
    F.count("*").alias("duree_min"),
    F.avg("lon").alias("lon"),
    F.avg("lat").alias("lat")
)
livraisons.cache()
print(livraisons.count())

273593


5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

In [16]:
ev = journal.filter(F.col("evenement").isin("livree", "echec"))
ev = ev.select(
    F.col("livreur_id").alias("ev_livreur"),
    F.to_date("horodatage").alias("ev_jour"),
    F.col("horodatage").alias("signature"),
    "evenement", "commande_id", "client_id"
)

# 1 minute de marge avant le debut et apres la fin
condition = (
    (F.col("livreur_id") == F.col("ev_livreur"))
    & (F.col("jour") == F.col("ev_jour"))
    & (F.col("signature") >= F.col("debut") - F.expr("INTERVAL 1 MINUTE"))
    & (F.col("signature") <= F.col("fin") + F.expr("INTERVAL 1 MINUTE"))
)

livraisons_ev = livraisons.join(ev, condition, "left").drop("ev_livreur", "ev_jour")
livraisons_ev.cache()

livraisons_ev.show(5)
print(livraisons_ev.count())

+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+-------------------+---------+-----------+---------+
|livreur_id|      jour|numero_livraison|vehicule_id|              debut|                fin|duree_min|               lon|               lat|          signature|evenement|commande_id|client_id|
+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+-------------------+---------+-----------+---------+
|      L002|2025-05-09|               0|        V03|2025-05-09 07:41:00|2025-05-09 08:23:00|       43|2.2770558372093013| 48.93673423255813|               NULL|     NULL|       NULL|     NULL|
|      L002|2025-05-09|               1|        V03|2025-05-09 10:37:00|2025-05-09 10:41:00|        5|2.3160694000000004|        48.8426472|2025-05-09 10:40:08|   livree|   C0072365|  CL01156|
|      L002|2025-05-09|            

307237


5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

In [17]:
rattaches = livraisons_ev.filter(F.col("commande_id").isNotNull()).select("commande_id", "signature")
pas_rattaches = ev.join(rattaches, ["commande_id", "signature"], "left_anti")
print(pas_rattaches.count())

0


5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

In [18]:
e = entrepots.select(F.col("lon").alias("e_lon"), F.col("lat").alias("e_lat"))
dist = livraisons.crossJoin(e).withColumn("d", haversine("lon", "lat", "e_lon", "e_lat"))
dist = dist.groupBy("livreur_id", "jour", "numero_livraison").agg(F.min("d").alias("dist_entrepot"))

nb = livraisons_ev.groupBy("livreur_id", "jour", "numero_livraison").agg(F.count("evenement").alias("nb_ev"))

types = nb.join(dist, ["livreur_id", "jour", "numero_livraison"])
types = types.withColumn("type",
    F.when(F.col("nb_ev") > 0, "livraison")
     .when(F.col("dist_entrepot") < 0.3, "entrepot")
     .otherwise("pause"))
types = types.select("livreur_id", "jour", "numero_livraison", "type")

livraisons_ev = livraisons_ev.join(types, ["livreur_id", "jour", "numero_livraison"])
livraisons_ev.cache()

# heure de debut en heures decimales (12.5 = 12h30)
res = livraisons.join(types, ["livreur_id", "jour", "numero_livraison"])
res = res.withColumn("h_debut", F.hour("debut") + F.minute("debut") / 60)
res.groupBy("type").agg(
    F.count("*").alias("nb"),
    F.round(F.avg("duree_min"), 1).alias("duree_moyenne"),
    F.round(F.avg("h_debut"), 2).alias("heure_moyenne_debut")
).show()

+---------+------+-------------+-------------------+
|     type|    nb|duree_moyenne|heure_moyenne_debut|
+---------+------+-------------+-------------------+
|    pause|  4967|         37.5|              12.35|
|livraison|246258|          8.3|              12.19|
| entrepot| 22368|         27.2|              11.68|
+---------+------+-------------+-------------------+



5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

In [19]:
liv = livraisons_ev.filter(F.col("type") == "livraison")
liv = liv.groupBy("livreur_id", "jour", "numero_livraison").agg(
    F.count("evenement").alias("nb_evenements"),
    F.countDistinct("client_id").alias("nb_clients")
)

liv.groupBy("nb_evenements").count().orderBy("nb_evenements").show()
print("au moins 2 clients :", liv.filter(F.col("nb_clients") >= 2).count())

+-------------+------+
|nb_evenements| count|
+-------------+------+
|            1|217538|
|            2| 24873|
|            3|  3053|
|            4|   592|
|            5|   155|
|            6|    29|
|            7|     8|
|            8|     5|
|            9|     4|
|           10|     1|
+-------------+------+



au moins 2 clients : 12536


5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

In [20]:
w_ev = Window.partitionBy("livreur_id", "jour", "numero_livraison").orderBy("signature")

livraisons_ev = livraisons_ev.withColumn("prec", F.lag("signature").over(w_ev))
livraisons_ev = livraisons_ev.withColumn("prec", F.coalesce("prec", "debut"))
livraisons_ev = livraisons_ev.withColumn("duree_ev_min",
    (F.unix_timestamp("signature") - F.unix_timestamp("prec")) / 60)
# avec la marge d'1 min la signature peut etre juste avant le debut -> on met 0
livraisons_ev = livraisons_ev.withColumn("duree_ev_min",
    F.when(F.col("duree_ev_min") < 0, 0).otherwise(F.col("duree_ev_min")))
livraisons_ev = livraisons_ev.drop("prec")

livraisons_ev.filter(F.col("evenement").isNotNull()).groupBy("evenement").agg(
    F.round(F.avg("duree_ev_min"), 2).alias("duree_moyenne")).show()

+---------+-------------+
|evenement|duree_moyenne|
+---------+-------------+
|    echec|         2.02|
|   livree|         4.24|
+---------+-------------+



5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

In [21]:
jours = gps.groupBy("livreur_id").agg(F.countDistinct("jour").alias("jours"))

stats = livraisons_ev.groupBy("livreur_id").agg(
    F.sum(F.when(F.col("evenement") == "livree", 1).otherwise(0)).alias("signees"),
    F.sum(F.when(F.col("evenement") == "echec", 1).otherwise(0)).alias("echecs")
)
stats = stats.join(jours, "livreur_id")
stats = stats.withColumn("signees_par_jour", F.round(F.col("signees") / F.col("jours"), 2))

stats.orderBy(F.col("signees_par_jour").desc()).show(10)

+----------+-------+------+-----+----------------+
|livreur_id|signees|echecs|jours|signees_par_jour|
+----------+-------+------+-----+----------------+
|      L076|    468|    24|   15|            31.2|
|      L089|    804|    42|   26|           30.92|
|      L070|    543|    28|   18|           30.17|
|      L086|    713|    26|   24|           29.71|
|      L081|    615|    18|   21|           29.29|
|      L072|    490|    21|   17|           28.82|
|      L079|    459|    16|   16|           28.69|
|      L085|    715|    36|   25|            28.6|
|      L080|    457|    18|   16|           28.56|
|      L082|    451|    19|   16|           28.19|
+----------+-------+------+-----+----------------+
only showing top 10 rows



## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

7.2) Panier moyen : montant moyen d'une commande.

7.3) Panier médian.

7.4) Top 10 des produits par chiffre d'affaires.

7.5) Top 10 des produits par quantité vendue.

7.6) Top 10 des clients par dépense sur l'année.

7.7) Top 10 des clients par nombre de commandes.

## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

8.4) Motifs d'échec : nombre d'échecs par motif.

8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*